# Data Preparation

This notebook implements various data preparation steps, addressing the following topics:

- Data Integration
- Assessment of Dimensions of Data Quality
- Redundancy Removal
- Missing Data Handling
- Outlier Detection and Handling
- Data Transformation
- Feature Engineering
- Sampling for Domain-Specific Purposes
- Sampling for Development
- Handling Imbalanced Data
- Feature Selection

In [2]:

# Import required libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

# Load the data
tables = {
    "awards_players": pd.read_csv('../data/development_data/awards_players.csv'),
    "coaches": pd.read_csv('../data/development_data/coaches.csv'),
    "players": pd.read_csv('../data/development_data/players.csv'),
    "players_teams": pd.read_csv('../data/development_data/players_teams.csv'),
    "series_post": pd.read_csv('../data/development_data/series_post.csv'),
    "teams": pd.read_csv('../data/development_data/teams.csv'),
    "teams_post": pd.read_csv('../data/development_data/teams_post.csv'),
}


## Data Integration

In [3]:

# Conversion of formats and entity matching
# Example: Convert height from inches to centimeters in players
if 'height' in tables['players'].columns:
    tables['players']['height_cm'] = tables['players']['height'] * 2.54

# Match playerID across tables to ensure consistency
players_ids = set(tables['players']['bioID'])
tables['awards_players'] = tables['awards_players'][tables['awards_players']['playerID'].isin(players_ids)]


## Assessment of Dimensions of Data Quality

In [4]:

# Assessment of six dimensions of data quality
dimensions = {}
for name, df in tables.items():
    dimensions[name] = {
        'Accuracy': None,  # Domain-specific checks
        'Completeness': df.notnull().mean() * 100,
        'Consistency': None,  # Verify matching IDs or other constraints
        'Uniqueness': df.duplicated().mean() * 100,
        'Timeliness': None,  # Depends on domain timestamps
        'Validity': None  # Check ranges or formats
    }
    print(f"Data Quality Dimensions for {name}:", dimensions[name])


Data Quality Dimensions for awards_players: {'Accuracy': None, 'Completeness': playerID    100.0
award       100.0
year        100.0
lgID        100.0
dtype: float64, 'Consistency': None, 'Uniqueness': np.float64(0.0), 'Timeliness': None, 'Validity': None}
Data Quality Dimensions for coaches: {'Accuracy': None, 'Completeness': coachID        100.0
year           100.0
tmID           100.0
lgID           100.0
stint          100.0
won            100.0
lost           100.0
post_wins      100.0
post_losses    100.0
dtype: float64, 'Consistency': None, 'Uniqueness': np.float64(0.0), 'Timeliness': None, 'Validity': None}
Data Quality Dimensions for players: {'Accuracy': None, 'Completeness': bioID           100.000000
pos              91.265398
firstseason     100.000000
lastseason      100.000000
height          100.000000
weight          100.000000
college          81.298992
collegeOther      1.231803
birthDate       100.000000
deathDate       100.000000
height_cm       100.000000
dtype: 

## Redundancy Removal

In [5]:

# Remove redundant columns based on high correlation
correlation = tables['teams'].select_dtypes(include=['number']).corr()
high_correlation = correlation.abs() > 0.9
columns_to_drop = [col for col in correlation.columns if any(high_correlation[col])]
tables['teams'] = tables['teams'].drop(columns=columns_to_drop, errors='ignore')


## Missing Data Handling

In [6]:

# Use regression to handle missing values
missing = tables['players']['height'].isnull()
if not tables['players'][missing][['weight']].dropna().empty:
    train = tables['players'].dropna(subset=['height', 'weight'])
    model = RandomForestRegressor(random_state=42)
    model.fit(train[['weight']], train['height'])
    tables['players'].loc[missing, 'height'] = model.predict(tables['players'][missing][['weight']].dropna())


## Outlier Detection and Handling

In [7]:

# Detect and handle outliers using Z-scores
from scipy.stats import zscore
z_scores = zscore(tables['players']['height'].dropna())
outliers = abs(z_scores) > 3
tables['players'] = tables['players'][~outliers]


## Data Transformation

In [8]:

# Apply complex scaling transformations
scaler = StandardScaler()
tables['players']['height_scaled'] = scaler.fit_transform(tables['players'][['height']])


## Feature Engineering

In [14]:
# Add features combining domain knowledge and aggregations
if 'won' in tables['teams'].columns and 'GP' in tables['teams'].columns:
    tables['teams']['win_rate'] = tables['teams']['won'] / tables['teams']['GP']
else:
    print("The required columns ('won' or 'GP') are missing in the 'teams' table.")


The required columns ('won' or 'GP') are missing in the 'teams' table.


## Sampling for Domain-Specific Purposes

In [15]:

# Select players who played more than 50 games
sample = tables['players_teams'][tables['players_teams']['GP'] > 50]


## Sampling for Development

In [16]:

# Start with small sample and grow
small_sample = tables['players'].sample(frac=0.1, random_state=42)
large_sample = tables['players'].sample(frac=0.5, random_state=42)


## Handling Imbalanced Data

In [17]:

# Handle imbalanced data using SMOTE
X = tables['players_teams'][['GP', 'GS', 'minutes']].dropna()
y = tables['players_teams']['points'] > 10  # Binary classification target
smote = SMOTE()
X_resampled, y_resampled = smote.fit_resample(X, y)


## Feature Selection

In [18]:

# Combine filter and wrapper methods
selector = SelectKBest(f_classif, k=2)
X_filtered = selector.fit_transform(X_resampled, y_resampled)

model = LogisticRegression()
rfe = RFE(model, n_features_to_select=2)
X_wrapper_selected = rfe.fit_transform(X_filtered, y_resampled)
